# 01 - Baseline: YOLO Single-Stage Digit Detection

Direct digit detection using YOLOv11. Each digit is a separate bounding box (classes 0-9).
Sorted left-to-right -> reconstructed meter reading.

In [1]:
import sys, subprocess
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --- Environment detection ---
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(
            ["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"],
            check=True,
        )
    BRANCH = "develop"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ultralytics", "albumentations", "rapidfuzz", "shapely"],
        check=True,
    )
    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..").resolve()
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0  # Windows CUDA: 0 for stability

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import json
import time
import csv
import torch
import cv2
import numpy as np
from datetime import datetime
from ultralytics import YOLO

from models.data.unified_loader import load_utility_meter_dataset, load_water_meter_dataset
from models.metrics.evaluation import full_string_accuracy, per_digit_accuracy, character_error_rate
from models.utils.visualization import draw_digit_bboxes

print(f"ROOT:        {ROOT}")
print(f"DATA_ROOT:   {DATA_ROOT}")
print(f"WEIGHTS_BASE:{WEIGHTS_BASE}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"WORKERS:     {WORKERS}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


ROOT: C:\Users\alike\WaterMeterCV
CUDA available: True
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ===================================================================
# MODEL SIZE - change this to switch between nano/small/medium:
#   "yolo11n" - nano  (~2.6M params, ~15 ms/img)  <- start here
#   "yolo11s" - small (~9.6M params, ~45 ms/img)  <- better accuracy
#   "yolo11m" - medium (~20M params, ~120 ms/img) <- max quality, batch down
# ===================================================================
MODEL_SIZE = "yolo11n"

# Paths
DATASET_PATH = ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11"
WM_PATH = ROOT / "WaterMetricsDATA/waterMeterDataset/WaterMeters"
DATA_YAML = DATASET_PATH / "data.yaml"
WEIGHTS_DIR = ROOT / "models/weights/baseline_yolo"
RESULTS_DIR = ROOT / "results"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
EPOCHS = 50
BATCH_SIZE = 16  # reduce to 8-12 for yolo11m if OOM
IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATIENCE = 10
WORKERS = 0  # Windows CUDA stability: avoids pin-memory worker crash

RUN_NAME = f"{MODEL_SIZE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Model: {MODEL_SIZE}")
print(f"Device: {DEVICE}")
print(f"Workers: {WORKERS}")
print(f"Run name: {RUN_NAME}")

Model: yolo11n
Device: cuda
Workers: 0
Run name: yolo11n_20260410_230900


In [3]:
# Load original data.yaml
with open(DATA_YAML) as f:
    data_config = yaml.safe_load(f)

# Fix paths: original uses ../train/images (relative to yaml),
# which breaks depending on CWD. Set absolute path to be safe.
data_config['path'] = str(DATASET_PATH)
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images'

FIXED_YAML = WEIGHTS_DIR / "data.yaml"
with open(FIXED_YAML, 'w') as f:
    yaml.dump(data_config, f)

print(f"Classes ({data_config['nc']}): {data_config['names']}")
print(f"Fixed data.yaml saved to {FIXED_YAML}\n")

# Count images per split
for split_name, split_dir in [("train", "train"), ("valid", "valid"), ("test", "test")]:
    img_dir = DATASET_PATH / split_dir / "images"
    count = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    print(f"  {split_name}: {count} images")

Classes (14): ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'Reading Digit', 'black', 'red', 'white']
Fixed data.yaml saved to C:\Users\alike\WaterMeterCV\models\weights\baseline_yolo\data.yaml

  train: 1552 images
  valid: 194 images
  test: 193 images


## Training

In [4]:
model = YOLO(f"{MODEL_SIZE}.pt")

# Clear stale allocations before training retry
if torch.cuda.is_available():
    torch.cuda.empty_cache()

results = model.train(
    data=str(FIXED_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(WEIGHTS_DIR),
    name=RUN_NAME,
    device=DEVICE,
    patience=PATIENCE,
    workers=WORKERS,
    save=True,
)

print(f"\nTraining complete. Best weights: {WEIGHTS_DIR / RUN_NAME / 'weights' / 'best.pt'}")

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.13.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\alike\WaterMeterCV\models\weights\baseline_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937

## Evaluation

In [9]:
# Load best model
best_weights = WEIGHTS_DIR / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(str(best_weights))

# YOLO built-in validation -> mAP
val_results = best_model.val(data=str(FIXED_YAML), split="test")
mAP50 = val_results.box.map50
mAP50_95 = val_results.box.map

print(f"mAP50:    {mAP50:.4f}")
print(f"mAP50-95: {mAP50_95:.4f}")
print("Per-class AP50:")
for i, name in enumerate(data_config["names"]):
    if i < len(val_results.box.ap50):
        print(f"  {name}: {val_results.box.ap50[i]:.4f}")


def predict_value(model, image_path):
    """Run YOLO and reconstruct digit string from detections.

    Returns (predicted_string, raw_result).
    """
    result = model.predict(str(image_path), verbose=False)[0]
    if result.boxes is not None and len(result.boxes) > 0:
        digit_mask = result.boxes.cls <= 9
        digit_boxes = result.boxes[digit_mask]
        if len(digit_boxes) > 0:
            sorted_idx = digit_boxes.xywh[:, 0].argsort()
            pred_str = "".join(str(int(digit_boxes.cls[i].item())) for i in sorted_idx)
            return pred_str, result
    return "", result


# Custom metrics + inference timing
test_samples = load_utility_meter_dataset(DATASET_PATH, split="test")

predictions = []
ground_truths = []

t_start = time.perf_counter()
for sample in test_samples:
    pred_str, _ = predict_value(best_model, sample.image_path)
    predictions.append(pred_str)

    gt_str = ""
    if sample.value is not None:
        gt_str = str(int(sample.value)) if sample.value == int(sample.value) else str(sample.value)
    ground_truths.append(gt_str)
t_total_ms = (time.perf_counter() - t_start) * 1000
avg_inference_ms = t_total_ms / len(test_samples) if test_samples else 0.0

# Compute metrics
fsa = full_string_accuracy(predictions, ground_truths)

pda_pairs = [(p, g) for p, g in zip(predictions, ground_truths) if g]
pda = sum(per_digit_accuracy(p, g) for p, g in pda_pairs) / len(pda_pairs) if pda_pairs else 0.0

cer_pairs = [(p, g) for p, g in zip(predictions, ground_truths) if g]
cer = sum(character_error_rate(p, g) for p, g in cer_pairs) / len(cer_pairs) if cer_pairs else 0.0

combined = 0.8 * mAP50 + 0.2 * fsa

print(f"\n{'='*50}")
print(f"Full-string accuracy: {fsa:.4f}")
print(f"Per-digit accuracy:   {pda:.4f}")
print(f"CER:                  {cer:.4f}")
print(f"Avg inference:        {avg_inference_ms:.1f} ms/image")
print(f"{'='*50}")
print(f"Combined Score (0.8*mAP50 + 0.2*FSA): {combined:.4f}")

Ultralytics 8.4.33  Python-3.13.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
YOLO11n summary (fused): 101 layers, 2,584,882 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2125.1813.7 MB/s, size: 565.4 KB)
val: Scanning C:\Users\alike\WaterMeterCV\WaterMetricsDATA\utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11\test\labels.cache... 193 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 193/193 67.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 4.1it/s 3.2s0.1s
                   all        193       1500      0.729      0.548      0.618      0.335
                     0        173        491       0.88      0.733      0.774      0.452
                     1        106        141       0.66      0.695      0.674       0.32
                     2         88        110      0.781      0.618       0.68      0.407
                     

## Predictions

In [10]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, sample in zip(axes.flat, test_samples[:8]):
    img = cv2.imread(str(sample.image_path))
    if img is None:
        ax.set_title("not found")
        ax.axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img = img.shape[:2]

    pred_str, result = predict_value(best_model, sample.image_path)

    # Draw predicted digit bboxes
    if result.boxes is not None and len(result.boxes) > 0:
        digit_mask = result.boxes.cls <= 9
        digit_boxes = result.boxes[digit_mask]
        if len(digit_boxes) > 0:
            bboxes = []
            for i in range(len(digit_boxes)):
                cls_id = int(digit_boxes.cls[i].item())
                cx = digit_boxes.xywh[i, 0].item() / w_img
                cy = digit_boxes.xywh[i, 1].item() / h_img
                bw = digit_boxes.xywh[i, 2].item() / w_img
                bh = digit_boxes.xywh[i, 3].item() / h_img
                bboxes.append((cls_id, cx, cy, bw, bh))
            img = draw_digit_bboxes(img, bboxes)

    gt_str = ""
    if sample.value is not None:
        gt_str = str(int(sample.value)) if sample.value == int(sample.value) else str(sample.value)

    ax.imshow(img)
    ax.set_title(f"GT={gt_str or '-'} | Pred={pred_str or '-'}", fontsize=10)
    ax.axis("off")

plt.suptitle(f"Baseline YOLO ({MODEL_SIZE}) - Test Predictions", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_predictions.png", dpi=150)
plt.close()
print("Saved to results/baseline_predictions.png")

Saved to results/baseline_predictions.png


## Cross-Dataset Evaluation (waterMeterDataset)

Per-digit accuracy only - full-string skipped because WM ground truth has decimals
(e.g. 78.677) while YOLO predicts digit sequences only (78677). Decimal handling deferred.

In [11]:
wm_samples = load_water_meter_dataset(WM_PATH)

wm_pda_scores = []
for sample in wm_samples:
    pred_str, _ = predict_value(best_model, sample.image_path)

    # Strip decimal point from GT for digit-only comparison
    gt_str = ""
    if sample.value is not None:
        gt_str = str(sample.value).replace(".", "")

    if gt_str:
        wm_pda_scores.append(per_digit_accuracy(pred_str, gt_str))

wm_pda = sum(wm_pda_scores) / len(wm_pda_scores) if wm_pda_scores else 0.0
print(f"waterMeterDataset - per-digit accuracy: {wm_pda:.4f}  (N={len(wm_pda_scores)})")

waterMeterDataset - per-digit accuracy: 0.0636  (N=1244)


## Save Results

In [12]:
# Save full metrics to JSON
metrics = {
    "model_size": MODEL_SIZE,
    "run_name": RUN_NAME,
    "primary_eval": {
        "mAP50": round(float(mAP50), 4),
        "mAP50_95": round(float(mAP50_95), 4),
        "full_string_accuracy": round(fsa, 4),
        "per_digit_accuracy": round(pda, 4),
        "CER": round(cer, 4),
        "combined_score": round(combined, 4),
        "avg_inference_ms": round(avg_inference_ms, 1),
    },
    "cross_dataset_eval": {
        "wm_per_digit_accuracy": round(wm_pda, 4),
    },
    "config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "patience": PATIENCE,
    },
    "run_date": datetime.now().isoformat(),
}

with open(RESULTS_DIR / "baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Append to comparison CSV (one row per model size run)
csv_path = RESULTS_DIR / "baseline_comparison.csv"
csv_exists = csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    if not csv_exists:
        writer.writerow([
            "model_size", "mAP50", "mAP50_95", "full_string_acc",
            "per_digit_acc", "CER", "combined_score", "inference_ms", "run_date",
        ])
    writer.writerow([
        MODEL_SIZE,
        round(float(mAP50), 4), round(float(mAP50_95), 4),
        round(fsa, 4), round(pda, 4), round(cer, 4),
        round(combined, 4), round(avg_inference_ms, 1),
        datetime.now().strftime("%Y-%m-%d %H:%M"),
    ])

print(f"Metrics -> {RESULTS_DIR / 'baseline_metrics.json'}")
print(f"CSV    -> {csv_path}")

Metrics -> C:\Users\alike\WaterMeterCV\results\baseline_metrics.json
CSV    -> C:\Users\alike\WaterMeterCV\results\baseline_comparison.csv


## Conclusions

Базовый прогон завершен (модель: **yolo11n**, запуск: **yolo11n_20260410_230900**).

- **Combined Score:** 0.5036
- **mAP50 / mAP50-95:** 0.6179 / 0.3351
- **Full-string accuracy:** 0.0466
- **Per-digit accuracy:** 0.1014
- **CER:** 0.8495
- **Inference:** 23.7 ms/image
- **Cross-dataset (WM per-digit):** 0.0636

### Что означают эти результаты

- Качество детекции умеренное (mAP50 ~0.62), но качество восстановления полного показания низкое (очень низкие full-string и per-digit accuracy).
- Модель часто находит области с цифрами, но реконструкция корректной последовательности остается нестабильной на сложных кадрах.
- Перенос на другой датасет слабый, текущий baseline плохо обобщается на `waterMeterDataset`.

### Важное наблюдение по формату строки

- В части примеров стоящие впереди нули (ведущие нули) в целевой записи фактически не учитываются (например, `00482` и `482`).
- В нашей реконструкции строки ведущие нули учитывались, поэтому в ряде случаев появлялись формальные расхождения в показаниях по строке, хотя по сути цифры распознаны верно.
- Из-за этого full-string accuracy может быть занижен, а CER — завышен; в следующем прогоне лучше считать метрики в двух вариантах: raw и normalized (без ведущих нулей).

### Важное наблюдение по качеству разметки

Визуальная проверка `val_batch{x}_labels` показывает, что часть лейблов может быть не полностью согласована с поворотами изображений (некоторые счетчики выглядят повернутыми, а геометрия/порядок разметки местами выглядят так, как будто аннотация сделана до поворота).
Это может вносить шум в обучающие данные и ухудшать итоговое качество модели.

Также видно, что нейросеть хуже распознает повернутые фотографии: на таких примерах предсказания заметно менее стабильны, чем на фронтальных/горизонтальных снимках счетчиков.

### Следующий шаг

Переходить к ROI-first пайплайну и добавить устойчивость к поворотам:
- очистить и перепроверить разметку повернутых примеров в train/val
- усилить rotation-аугментации
- после очистки разметки повторить запуск на `yolo11s`